In [94]:
import pandas as pd                                                                 
import glob                                                                         
import re 

In [96]:
# Load and save as parquet                                                          
files = glob.glob('../data/raw/luka_*.csv')
                                            
dfs = []                                                                            
for f in files:                         
    year = re.search(r'(\d{4}_\d{2})', f).group(1)                                  
    df = pd.read_csv(f, skiprows=0)                                                 
                                    
    # BBRef CSVs have junk rows — drop rows where Rk is not numeric                 
    df = df[pd.to_numeric(df['Rk'], errors='coerce').notna()]      
    df['SEASON'] = year                                      
                                                                                    
    # Save parquet                                                                  
    out = f'../data/processed/luka_csv_{year}.parquet'                              
    df.to_parquet(out, index=False)                                                 
    print(f"Saved {out}")                   
    dfs.append(df)                                                                  
                
df_all = pd.concat(dfs, ignore_index=True)

Saved ../data/processed/luka_csv_2020_21.parquet
Saved ../data/processed/luka_csv_2018_19.parquet
Saved ../data/processed/luka_csv_2022_23.parquet
Saved ../data/processed/luka_csv_2024_25.parquet
Saved ../data/processed/luka_csv_2021_22.parquet
Saved ../data/processed/luka_csv_2023_24.parquet
Saved ../data/processed/luka_csv_2019_20.parquet


In [97]:
files = sorted(glob.glob('../data/raw/luka_*.csv'))                                 
                                                                                    
dfs = []                                                                            
for f in files:                                                                     
    year = re.search(r'(\d{4}_\d{2})', f).group(1)                                  
    df = pd.read_csv(f)                           
    df = df[pd.to_numeric(df['Rk'], errors='coerce').notna()].copy()                
    df.rename(columns={df.columns[5]: 'HOME_AWAY'}, inplace=True)   
    df['HOME'] = (df['HOME_AWAY'] != '@').astype(int)                               
    df['SEASON'] = year                              
    df['Date'] = pd.to_datetime(df['Date'])
    df['PTS'] = pd.to_numeric(df['PTS'], errors='coerce')                           
    df['TRB'] = pd.to_numeric(df['TRB'], errors='coerce')                           
    df['AST'] = pd.to_numeric(df['AST'], errors='coerce')                           
    df.to_parquet(f'../data/processed/luka_csv_{year}.parquet', index=False)        
    dfs.append(df)                                                                  
    print(f"{year}: {len(df)} games")   
                                                                                    
df_all = pd.concat(dfs,                                                             
ignore_index=True).sort_values('Date').reset_index(drop=True)                       
print(f"\nTotal: {len(df_all)} games")                                              
                                                                                    
# Basic analysis                            
print("\n--- Season Averages ---")                                                  
print(df_all.groupby('SEASON')[['PTS','TRB','AST']].mean().round(1))
                                                                                    
print("\n--- Home vs Away ---")         
print(df_all.groupby('HOME')[['PTS','TRB','AST']].mean().round(1))                  
                                                                                    
print("\n--- Overall Distribution ---")
print(df_all['PTS'].describe().round(1)) 

2018_19: 82 games
2019_20: 75 games
2020_21: 72 games
2021_22: 82 games
2022_23: 82 games
2023_24: 82 games
2024_25: 84 games

Total: 559 games

--- Season Averages ---
          PTS  TRB  AST
SEASON                 
2018_19  21.2  7.8  6.0
2019_20  28.8  9.4  8.8
2020_21  27.7  8.0  8.6
2021_22  28.4  9.1  8.7
2022_23  32.4  8.6  8.0
2023_24  33.9  9.2  9.8
2024_25  28.2  8.2  7.7

--- Home vs Away ---
       PTS  TRB  AST
HOME                
0     27.9  8.5  8.1
1     29.3  8.7  8.4

--- Overall Distribution ---
count    450.0
mean      28.6
std        9.0
min        0.0
25%       23.0
50%       28.5
75%       34.0
max       73.0
Name: PTS, dtype: float64
